In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import CCA
import muon as mu
import scanpy as sc
import scirpy as ir

import sys
sys.path.append("/ihome/ylee/yiz133/Code/Data processing/functions")
import importlib
import mdata_utils


In [ ]:
importlib.reload(mdata_utils)

In [ ]:
%cd "/ix1/ylee/Yifan_Zhang/Code_data/external"

In [ ]:
mdata = mu.read("merged_EAE_TCRemb.h5mu")
mdata

# Load features

In [ ]:
view_gene = mdata.obsm["X_pca_harmony"]

In [ ]:
arrs_tcr = []
for key, value in mdata.obsm.items():
    arrs_tcr.append(value)

view_tcr = np.concatenate(arrs_tcr, axis=1)

scaler = StandardScaler()
view_gene = scaler.fit_transform(view_gene)
view_tcr = scaler.fit_transform(view_tcr)

In [ ]:
labels = mdata.obs["state"]
label_bin = (mdata.obs["state"] == 'CNS').astype(int)

# CCA

In [ ]:
# Perform Canonical Correlation Analysis
cca = CCA(n_components=4)
view_gene_c, view_tcr_c = cca.fit_transform(view_gene, view_tcr)


In [ ]:
# weights
x_loadings_df = pd.DataFrame(cca.x_loadings_)
print(x_loadings_df.head())

# There is a corresponding matrix for Y
y_loadings_df = pd.DataFrame(cca.y_loadings_)

In [ ]:
# calculate the correlations

correlations = []
cv_list = []
for i in range(cca.x_loadings_.shape[1]):
  # Calculate the correlation coefficient between the i-th columns of X_c and Y_c
  # np.corrcoef returns a 2x2 matrix, the value we want is at position (0, 1)
  corr = np.corrcoef(view_gene_c[:, i], view_tcr_c[:, i])[0, 1]
  correlations.append(corr)

  mdata['gex'].obs['CV_score_'+str(i)] = view_gene_c[:,i]
  # mdata['tcr'].obs['CV_score_'+str(i)] = view_tcr_c[:,i]
  cv_list.append('CV_score_'+str(i))

# Print the results
print("Canonical Correlations:")
for i, corr in enumerate(correlations):
  print(f"  Correlation for Component {i+1}: {corr:.4f}")

In [ ]:
sc.pl.embedding(mdata["gex"], basis = 'X_umap_harmony', neighbors_key = 'neighbors_harmony', 
            color= cv_list, ncols=2,)

In [ ]:
abs_target = ['tissue', 'cell_type', 'state']
cv_thres_p = 1
cv_thres_n = -2

# Loop through CV_score_1 to CV_score_4
all_results = []

for i in range(1, 5):
    cv_name = f'CV_score_{i}'
    
    # Get positive and negative subsets
    mdata_sub_pos = mdata[(mdata['gex'].obs[cv_name] > cv_thres_p)]
    mdata_sub_neg = mdata[(mdata['gex'].obs[cv_name] < cv_thres_n)]
    
    # Filter for Spleen or CNS tissue only
    mdata_sub_pos_filtered = mdata_sub_pos[mdata_sub_pos['gex'].obs['tissue'].isin(['Spleen', 'CNS'])]
    mdata_sub_neg_filtered = mdata_sub_neg[mdata_sub_neg['gex'].obs['tissue'].isin(['Spleen', 'CNS'])]
    
    # Calculate percentages for positive subset
    percentage_dict = {'CV_component': cv_name}
    
    for col in abs_target:
        counts = mdata_sub_pos_filtered['gex'].obs[col].value_counts()
        percentages = (counts / len(mdata_sub_pos_filtered) * 100).round(2)
        for category, pct in percentages.items():
            col_name = f'pos_{category}'
            percentage_dict[col_name] = pct
    
    # Calculate percentages for negative subset
    for col in abs_target:
        counts = mdata_sub_neg_filtered['gex'].obs[col].value_counts()
        percentages = (counts / len(mdata_sub_neg_filtered) * 100).round(2)
        for category, pct in percentages.items():
            col_name = f'neg_{category}'
            percentage_dict[col_name] = pct
    
    all_results.append(percentage_dict)

df_CV = pd.DataFrame(all_results)
df_CV


In [ ]:
# ANOVA tests: comparing all groups within each abs_target
from scipy.stats import f_oneway

anova_results = []

for i in range(0,4):
    cv_name = f'CV_score_{i}'
    
    for col in abs_target:
        # Get unique categories
        categories = mdata['gex'].obs[col].dropna().unique()
        
        # Collect all groups for this variable
        groups = []
        for cat in categories:
            group_data = mdata['gex'].obs[mdata['gex'].obs[col] == cat][cv_name].dropna().values
            if len(group_data) > 0:
                groups.append(group_data)
        
        # Perform ANOVA comparing all groups
        if len(groups) > 1:
            f_stat, p_value = f_oneway(*groups)
            
            anova_results.append({
                'CV_component': cv_name,
                'variable': col,
                'n_groups': len(groups),
                'F_statistic': round(f_stat, 4),
                'p_value': f'{p_value:.4e}',
                'significant': 'Yes' if p_value < 0.05 else 'No'
            })

df_anova = pd.DataFrame(anova_results)
df_anova


In [ ]:
# Calculate Pearson correlation between marker genes and CV scores
from scipy.stats import pearsonr

# Get gene expression data
adata = mdata['gex']

# Store results
corr_results = []

for i in range(4):
    cv_name = f'CV_score_{i}'
    cv_scores = adata.obs[cv_name].values
    
    for gene in markers:
        if gene in adata.var_names:
            # Get gene expression values
            gene_expr = adata[:, gene].X.toarray().flatten() if hasattr(adata[:, gene].X, 'toarray') else adata[:, gene].X.flatten()
            
            # Calculate Pearson correlation
            corr, p_value = pearsonr(gene_expr, cv_scores)
            
            corr_results.append({
                'CV_component': cv_name,
                'gene': gene,
                'pearson_r': round(corr, 4),
                'p_value': f'{p_value:.4e}',
                'significant': 'Yes' if p_value < 0.05 else 'No'
            })
        else:
            print(f"Warning: Gene '{gene}' not found in dataset")

df_gene_corr = pd.DataFrame(corr_results)
df_gene_corr


In [ ]:
# Randomly keep one cell per clone_id within each GSE subset
airr = mdata['airr']

print(f"Original number of cells: {airr.n_obs}")

# Store indices to keep
indices_to_keep = []

# Group by GSE
for gse in airr.obs['GSE'].unique():
    # Get cells for this GSE
    gse_mask = airr.obs['GSE'] == gse
    gse_obs = airr.obs[gse_mask]
    
    print(f"\nGSE: {gse} - Original cells: {len(gse_obs)}")
    
    # For each clone_id in this GSE, randomly select one cell
    for clone_id in gse_obs['clone_id'].unique():
        clone_cells = gse_obs[gse_obs['clone_id'] == clone_id]
        # Randomly sample one cell from this clone
        selected_idx = clone_cells.sample(n=1, random_state=42).index[0]
        indices_to_keep.append(selected_idx)
    
    print(f"GSE: {gse} - After sampling: {len(gse_obs['clone_id'].unique())} cells (one per clone)")

# Create subset with selected cells
mdata_subset = mdata[indices_to_keep].copy()

print(f"\n{'='*60}")
print(f"Total cells after sampling: {mdata_subset['airr'].n_obs}")
print(f"Reduction: {airr.n_obs} -> {mdata_subset['airr'].n_obs} cells")


In [ ]:
# Logistic Regression: predict CNS vs rest using TCR features
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

# Create binary label: CNS vs rest
y = (mdata['gex'].obs['tissue'] == 'CNS').astype(int)

# Split data
X_train, X_test, y_train, y_test = train_test_split(view_tcr, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training set CNS ratio: {y_train.sum() / len(y_train):.2%}")
print(f"Test set CNS ratio: {y_test.sum() / len(y_test):.2%}")

# Apply RandomOverSampler to balance classes
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)
X_train_resampled, y_train_resampled = ros.fit_resample(X_train, y_train)

print(f"\nAfter oversampling:")
print(f"Training set: {X_train_resampled.shape[0]} samples")
print(f"Training set CNS ratio: {y_train_resampled.sum() / len(y_train_resampled):.2%}")

# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_resampled, y_train_resampled)

# Predictions
y_pred = lr_model.predict(X_test)
y_pred_proba = lr_model.predict_proba(X_test)[:, 1]

# Evaluation
print(f"\n{'='*60}")
print("Performance on Test Set:")
print(f"{'='*60}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-CNS', 'CNS']))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
# Plot distribution of decision function for both classes
decision_scores = lr_model.decision_function(X_test)

# Separate scores by class
scores_non_cns = decision_scores[y_test == 0]
scores_cns = decision_scores[y_test == 1]

# Create the plot
plt.figure(figsize=(10, 6))
plt.hist(scores_non_cns, bins=50, alpha=0.6, label='Non-CNS', color='blue', edgecolor='black')
plt.hist(scores_cns, bins=50, alpha=0.6, label='CNS', color='red', edgecolor='black')
plt.axvline(x=0, color='green', linestyle='--', linewidth=2, label='Decision boundary')
plt.xlabel('Decision Function Score', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Decision Function Scores by Class', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print statistics
print(f"Non-CNS scores - Mean: {scores_non_cns.mean():.4f}, Std: {scores_non_cns.std():.4f}")
print(f"CNS scores - Mean: {scores_cns.mean():.4f}, Std: {scores_cns.std():.4f}")
